In [ ]:
import pandas as pd

In [ ]:
#Load Data
customers = pd.read_csv('../data/raw/customers.csv')
offers = pd.read_csv('../data/raw/offers.csv')
events = pd.read_csv('../data/raw/events.csv')
data_dic = pd.read_csv('../data/raw/data_dictionary.csv')


In [ ]:
customers.head()

In [ ]:

customers.info()
customers.describe()


In [ ]:
offers.head()

In [ ]:

offers.info()
offers.describe()

In [ ]:
events.head()

In [ ]:
events.info()
events.describe()

# Data Cleaning
## Clean Customer Table

In customer table, we could see that when gender and income are null then age is 118. This likely represents missing or unknown customer data, not real age. So we replace 118 to NA.

In [ ]:
customers['age'] = customers['age'].replace(118, pd.NA)


We also handle missing value of gender and income

In [ ]:
customers.isnull().sum()

In [ ]:
customers['gender'] = customers['gender'].fillna('Unknown')

In [ ]:
customers['income'] = customers['income'].fillna(customers['income'].median())

We also convert data type of became_member_on to date

In [ ]:
customers['became_member_on'] = pd.to_datetime(customers['became_member_on'], format='%Y%m%d')
customers.head()

## Clean Events Table
The events dataset contains customer activities such as receiving, viewing, and completing offers, as well as transactions. However, the value column is semi-structured and stores different information depending on the event type. Offer-related events contain an offer id, while transaction events contain an amount.

In [ ]:
events['value'].head()

In [ ]:
events['event'].value_counts()

In [ ]:
events[events['event'] == 'transaction']['value'].head()

In [ ]:
# Convert value column from string to dictionary

import ast
events['value'] = events['value'].apply(ast.literal_eval)



In [ ]:
# extract relevent fields into seperate columns

events['offer_id'] = events['value'].apply(lambda x: x.get('offer id'))
events['amount'] = events['value'].apply(lambda x: x.get('amount'))

In [ ]:
# drop the origianl value column
events.drop('value', axis =1, inplace = True)



In [ ]:
events.head()

To make the data usable for analysis, the value column was first converted from a string into a dictionary format. Then, relevant fields were extracted into separate columns (offer_id and amount), allowing each type of information to be clearly represented. This transformation standardizes the dataset, making it possible to link events to offers, analyze customer spending, and track the full customer journey effectively.

## Check Duplicate values

In [ ]:
customers.duplicated().sum()

In [ ]:
events.duplicated().sum()

In [ ]:
offers.duplicated().sum()

In [ ]:
events = events.drop_duplicates()
events.duplicated().sum()

The events dataset contained 2,962 fully duplicated records identified through full-row comparison. Since each row represents a unique customer action at a specific time, identical records are unlikely to be valid. For example, if a customer has two identical entries for a transaction at the same timestamp with the same amount, counting both would incorrectly double the revenue. Therefore, these duplicates were removed to prevent double counting and ensure accurate KPI calculation

## convert time into days
Time column is in hours since the start of the experiment. So, we convert it to days as offers have duration in days and it is easier to compare offer duration and completion time.


In [ ]:
events['time_days'] = events['time']/24
events.tail()

## Clean Offer Table
Channels column has a list stored as text which is not directly usable in analysis. So, we transform the multi-valued channels field into binary indicator columns to enable channel-level performance analysis i.e one-hot encoding.

In [ ]:
# Convert string to list
offers['channels'] = offers['channels'].apply(ast.literal_eval)
offers['channels']



In [ ]:
# create dummy columns
channels_dummies = offers['channels'].explode().str.get_dummies().groupby(level=0).max()
offers = pd.concat([offers, channels_dummies], axis=1)
offers.head()


In [ ]:
## Merge data

events_offers = events.merge(offers, how='left', on='offer_id')
full_data = events_offers.merge(customers, how='left', on='customer_id')
full_data.head()





In [ ]:
full_data.info()

In [ ]:
## Save Clean Data
customers.to_csv('../data/cleaned/customers_clean.csv', index=False)
events.to_csv('../data/cleaned/events_clean.csv', index=False)
offers.to_csv('../data/cleaned/offers_clean.csv', index=False)
full_data.to_csv('../data/cleaned/full_data.csv', index=False)